# Phase 9 — آزمون کوتاه اصلاح تشخیص

این دفترچه فقط ۱۰ نمونهٔ **توسعه** را با شواهد ذخیره‌شده آزمایش می‌کند؛ ارزیابی کامل یا نتیجهٔ held-out نیست. حداکثر ۲۰ فراخوانی رایگان مدل (با یک تلاش مجدد در هر نمونه)، بدون Neo4j و Docker.

Runtime → Change runtime type → T4 GPU. سلول‌ها را به‌ترتیب اجرا کنید. فقط فایل `incidentgraph-phase9-repair.bundle` را آپلود کنید؛ ZIP قدیمی لازم نیست. در پایان `phase9-repair-probe.zip` را برگردانید.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path
from google.colab import files

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
assert gpu.returncode == 0, 'Select a free T4 GPU runtime first.'
print(gpu.stdout.strip())
print('Upload ONLY incidentgraph-phase9-repair.bundle')
uploaded = files.upload()
bundles = [(name, data) for name, data in uploaded.items() if name.endswith('.bundle')]
assert len(bundles) == 1, 'Select exactly one bundle.'
bundle = Path('/content/phase9-repair-upload.bundle')
bundle.write_bytes(bundles[0][1])
expected_hash = '29e5c07ee0130e439504411d8bf8a486e8969fb64de47c655f94baa742eb5eef'
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == expected_hash, (
    'This is not the new repair bundle. Upload incidentgraph-phase9-repair.bundle.'
)
print('Bundle verified.')


In [ ]:
workspace = Path(tempfile.mkdtemp(prefix='phase9-repair-', dir='/content'))
project = workspace / 'incidentgraph'
subprocess.run(['git', 'clone', '--branch', 'main', str(bundle), str(project)], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=project, text=True).strip()
assert revision == 'fe25806d5a5f9380e944133f298dbb7ce8d116b2', revision
assert (project / 'src/incidentgraph/phase9_repair.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--no-dev', '--python', '3.12'], cwd=project, check=True)
print('Repair code ready:', revision[:7])


In [ ]:
installer = workspace / 'install-ollama.sh'
urllib.request.urlretrieve('https://ollama.com/install.sh', installer)
subprocess.run(
    ['bash', str(installer)],
    env={**os.environ, 'OLLAMA_VERSION': '0.34.3'},
    check=True,
)
def server_version():
    with urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2) as response:
        return json.load(response)['version']

try:
    version = server_version()
except Exception:
    log = (workspace / 'ollama.log').open('w')
    subprocess.Popen(
        ['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434',
             'OLLAMA_NUM_PARALLEL': '1', 'OLLAMA_MAX_LOADED_MODELS': '1',
             'OLLAMA_CONTEXT_LENGTH': '8192'},
    )
    for _ in range(60):
        try:
            version = server_version()
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError((workspace / 'ollama.log').read_text()[-4000:])
assert version == '0.34.3', 'Use a fresh Colab runtime: expected Ollama 0.34.3, got ' + version
model_id = 'qwen3:4b-instruct-2507-q4_K_M'
subprocess.run(['ollama', 'pull', model_id], check=True)
with urllib.request.urlopen('http://127.0.0.1:11434/api/tags') as response:
    models = json.load(response)['models']
model = next(item for item in models if item['name'] == model_id)
assert model['digest'] == '0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0'
print('Pinned free model ready. Next cell runs only 10 development examples.')


In [ ]:
output_dir = workspace / 'phase9-repair-probe'
output_dir.mkdir(exist_ok=True)
try:
    subprocess.run([
        'uv', 'run', '--no-sync', 'python', '-m', 'incidentgraph.phase9_repair',
        '--output-dir', str(output_dir),
    ], cwd=project, check=True)
    print((output_dir / 'summary.json').read_text())
finally:
    # Preserve partial records and errors even when a later example fails.
    archive = shutil.make_archive(
        str(workspace / 'phase9-repair-probe'), 'zip', str(workspace), 'phase9-repair-probe'
    )
    print('Download this ZIP and send it back:', archive)
    files.download(archive)


پس از دانلود ZIP، اجرا تمام است. هیچ شارد ۰ تا ۱۱ یا اجرای ۱۲۰تایی در این دفترچه وجود ندارد. امتیاز این آزمون، نتیجهٔ توسعه است؛ بررسی نهایی فاز ۹ هنوز به اعتبارسنجی کامل، پوشش آزمون و rubric انسانی نیاز دارد.
